# PINN solution benchmark — Colab front end
Thin, reproducible driver for the audit + benchmark of the iB-function
traveling-wave catalogue (paper-5.pdf). All logic lives in the repo; this
notebook only installs, runs, displays, and packages.

In [ ]:
# 1. Install dependencies (upload or clone the pinn_solution_benchmark folder first)
%cd pinn_solution_benchmark
!pip -q install -r requirements.txt
!pip -q install -e .

In [ ]:
# 2. Check for the PDF (optional: the audit is self-contained, but the
#    inventory report references it). Upload paper-5.pdf next to the repo if desired.
from pathlib import Path
print('paper found:', Path('../paper-5.pdf').exists())

In [ ]:
# 3. (Optional) resume from previously saved state: upload a results/checkpoints
#    archive produced by cell 7 and unpack it. Completed jobs are skipped.
import pathlib
arch = pathlib.Path('benchmark_state.tar.gz')
if arch.exists():
    !tar xzf benchmark_state.tar.gz
    print('resumed state unpacked')

In [ ]:
# 4. Mathematical audit + unit tests + numerical validation
!python scripts/run_audit.py --config configs/audit.yaml
!python -m pytest tests/ -q
!python scripts/run_numerical.py --quick

In [ ]:
# 5. Choose smoke or pilot mode (pilot needs a GPU or several CPU-hours)
MODE = 'smoke'  # or 'pilot'
!python scripts/run_forward.py --config configs/{MODE}.yaml

In [ ]:
# 6. Display key tables and figures
import json, pandas as pd
from IPython.display import Image, Markdown, display
display(Markdown(open('reports/mathematical_audit.md').read()))
cat = pd.read_csv('data/verified_branches/branch_catalog.csv')
display(cat)
import glob
for f in sorted(glob.glob(f'reports/figures/forward_{MODE}/*.png'))[:6]:
    display(Image(f))

In [ ]:
# 7. Package results + checkpoints for download / later resumption
!tar czf benchmark_state.tar.gz results checkpoints data/verified_branches reports
from google.colab import files  # noqa
files.download('benchmark_state.tar.gz')